# Minesweeper

<pre>
    
</pre>

In [13]:
import random

class Tile: 
    def __init__(self, row, col):
        self._row = row
        self._col = col

        self._ismine = False
        self._isrevealed = False
        self._hint = 0
        self._isflagged = False

    def get_row(self):
        return self._row
    def set_row(self, row):
        self._row = row

    def get_col(self):
        return self._col
    def set_col(self, col):
        self._col = col

    def get_ismine(self):
        return self._ismine
    def set_ismine(self, ismine):
        self._ismine = ismine

    def get_isrevealed(self):
        return self._isrevealed
    def set_isrevealed(self, isrevealed):
        self._isrevealed = isrevealed

    def get_isflagged(self):
        return self._isflagged
    def set_isflagged(self, isflag):
        self._isflagged = isflag

    def get_hint(self):
        return self._hint
    def set_hint(self, hint):
        self._hint = hint


    
class Game:
    def __init__(self, board_size):
        self._board_size = board_size
        self._board = [[None for p in range(self._board_size)] for q in range(self._board_size)]
        self._win = False
        self._lose = False
        self._tiles_left = 0
        self._directions = {
            'up': (-1,0),
            'down': (1,0),
            'left': (0,-1),
            'right': (0,1),
            'upleft': (-1,-1),
            'upright': (-1,1),
            'downleft': (1,-1),
            'downright': (1,1),
        }
        self._symbols = {
            'urtile' : '[ ]',
            'urmine' : '[X]',
            'flag' : '[F]',
        }
    
    def get_tile(self, coord:tuple): # Done return Tile object
        (row,col) = coord
        return self._board[row][col]

    def coord_isvalid(self, coord:tuple): # Done 
        (row,col) = coord
        if 0 <= row < self._board_size and 0 <= col < self._board_size:
            return True
        return False

    def count_surr_mines(self, coord:tuple): # Done
        (row,col) = coord
        count = 0
        for direction in self._directions.values():
            y, x = direction
            new_row = row + y
            new_col = col + x
            if self.coord_isvalid((new_row, new_col)):
                if self._board[new_row][new_col].get_ismine():
                    count += 1
        return count

    def create_mines(self, y, x): # Done
        for row in range(self._board_size):
            for col in range(self._board_size):
                tile = Tile(row,col) 
                self._board[row][col] = tile
                if (row,col) != (y,x) and random.randint(0,10) in (1,2,3): # tile is not tile user clicked, roll to set mine
                    (self._board[row][col]).set_ismine(True)
                else: 
                    (self._board[row][col]).set_ismine(False)
                    self._tiles_left += 1
        print("Mines have been set")
        print(f'{self._tiles_left} Tile objects created')
        for row in range(self._board_size):
            for col in range(self._board_size): 
                # tile we are looking at is board[row][col]    
                self.get_tile((row,col)).set_hint(self.count_surr_mines((row, col)))
        print("Hints have been set")

    def display_board(self): # Done 
        line = f'     +{'-----+'*self._board_size}'
        output = '       '

        # create index row + first breakline
        for i in range(self._board_size): 
            output += f"{i}     "
        output+= '\n'+line+'\n'
        
        for row in range(self._board_size): 
            output += f" {row:2}  | "       # index column
            
            for col in range(self._board_size):
                tile = self.get_tile((row,col))

                if tile.get_isflagged(): # flags are displayed regardless of reveal status
                    output += f'{self._symbols['flag']} | '
                elif tile.get_isrevealed(): # for revealed tiles, display the hint 
                    if tile.get_hint() == 0:
                        output += '    | '
                    else: 
                        output += f' {tile.get_hint()}  | '
                else:                             # for unrevealed tile mines are X and nonmines are ' '
                    if self._lose and tile.get_ismine():
                        output += f'{self._symbols['urmine']} | '
                    else: 
                        output += f'{self._symbols['urtile']} | '

            output += '\n'+ line + '\n'
        print(output)
                    
    def get_user_coord(self): # Done returns tuple(row,col)
        try_count = 0
        while try_count < 10:
            print('Enter the coordinate in the form x,y: ')
            coord = input()
            print(f"Your input: {coord}")
            try:
                parts = coord.split(',')
                if len(parts) != 2:
                    raise ValueError
                col = int(parts[0])
                row = int(parts[1])
                if self.coord_isvalid((row,col)):
                    return (row, col)
                raise ValueError
            except ValueError:
                print("Please enter a valid coordinate.")
                try_count += 1
        print("Restart game.")
        return False

    def get_user_option(self): # Done returns "Dig" / "Flag" / "Unflag" / "Quit" 
        try_count = 0
        option_dict = {
            '1' : 'Flag',
            '2' : 'Unflag',
            '3' : 'Dig',
            '4' : 'Quit'
            }
        while try_count < 10:
            print('''
            Enter the option number: 
            1) Flag
            2) Unflag
            3) Dig
            4) Quit
                ''')
            try:
                user_option = input()
                if user_option not in ('1','2','3','4'):
                    raise ValueError
                user_option = option_dict[user_option]
                print(f"Your option: {user_option}")
                return user_option
            except ValueError:
                print("Please enter a valid option number. ")
                try_count += 1
        print("Restart game.")
        return False

    def reveal_tile(self, coord:tuple): # Done
        tile = self.get_tile(coord)
        tile.set_isrevealed(True)

    def dig(self,coord:tuple): # Done 
        # checks and terminating case
        if not self.coord_isvalid(coord):
            return False
        tile = self.get_tile(coord)
        if tile.get_isrevealed():
            return False
        elif tile.get_isflagged(): # cannot dig on tile
            print("Unable to dig on flag")
            return False
        
        tile_ismine = tile.get_ismine() 
        if tile_ismine: # if tile is mine: game over
            print("You have hit a mine!")
            self._lose = True  
        else:           # if tile is not mine: 
            tile.set_isrevealed(True)
            self._tiles_left -= 1 
            hint = tile.get_hint()

            if hint==0: # dig all surrounding mines
                for direction in self._directions.values():
                    (row,col) = coord
                    y, x = direction
                    new_row = row + y
                    new_col = col + x
                    self.dig((new_row,new_col))

    def place_flag(self,coord:tuple):
        tile = self.get_tile(coord)
        if tile.get_isflagged():
            print("Tile has already been flagged")
            return False
        elif tile.get_isrevealed():
            print("Tile cannot be flagged")
            return False
        else:
            tile.set_isflagged(True)

    def drop_flag(self,coord:tuple):
        tile = self.get_tile(coord)
        if tile.get_isflagged():
            tile.set_isflagged(False)
        else: 
            print(f"There is no flag on tile {coord}")
            return False
        
    def run_game(self): 
        while not self._lose and self._tiles_left > 0: 
            self.display_board () 
            coord = self.get_user_coord() # Done coord is tuple(row,col) 
            option = self.get_user_option() # Done option is "Dig" / "Flag" / "Quit" 
            if option == 'Dig':
                self.dig(coord) # Done
            elif option == 'Flag':
                self.place_flag(coord) ##################
            elif option == "Unflag":
                self.drop_flag(coord)
            elif option == 'Quit':
                self._lose = True
        self.display_board()
        print('''
============================================================================
============================================================================
#####     ###     #     #  #######        #######  #     #  #######  ####
#        #   #    ##   ##  #              #     #  #     #  #        #    #
#        #     #  # # # #  #              #     #  #     #  #        #    #
#  ####  #     #  #  #  #  #####          #     #  #     #  #####    # #
#     #  #######  #     #  #              #     #   #   #   #        #  #
#     #  #     #  #     #  #              #     #    # #    #        #   #
######   #     #  #     #  #######        #######     #     #######  #    #
============================================================================
============================================================================
''')  

    def start(self):
        print("Insert the cordinate of the tile in the form x,y")
        start_coord = (input()).split(',')
        # print(start_coord)
        try:
            col = int(start_coord[0])
            row = int(start_coord[1])
            # print(row, col)
            self.create_mines(row,col)
            self.run_game()
        except ValueError: 
            print("Enter a valid coordinate. ")
            
### test

def minesweeper():
    print(r"""
                 +===========================+
                 |       MINESWEEPER         |
                 +===========================+

                        Find the safe tiles.
                    Avoid the hidden mines!

                        HOW TO PLAY
                    -------------------
                    Enter coordinates: x,y

                    1) Flag a tile
                    2) Unflag a tile
                    3) Dig a tile
                    4) Quit game

                 +===========================+
                 |      GOOD LUCK!           |
                 +===========================+
    """)
    print("Please enter board size: ")
    boardsize = int(input())
    print(f"Boardsize: {boardsize}")
    if boardsize>=100: 
        print("Are you sure?")
        return
    g=Game(boardsize)
    g.start()



minesweeper()


                 +===========================+
                 |       MINESWEEPER         |
                 +===========================+

                        Find the safe tiles.
                    Avoid the hidden mines!

                        HOW TO PLAY
                    -------------------
                    Enter coordinates: x,y

                    1) Flag a tile
                    2) Unflag a tile
                    3) Dig a tile
                    4) Quit game

                 +===========================+
                 |      GOOD LUCK!           |
                 +===========================+
    
Please enter board size: 


Boardsize: 99
Insert the cordinate of the tile in the form x,y
Mines have been set
7071 Tile objects created
Hints have been set
       0     1     2     3     4     5     6     7     8     9     10     11     12     13     14     15     16     17     18     19     20     21     22     23     24     25     26     27     28     29     30     31     32     33     34     35     36     37     38     39     40     41     42     43     44     45     46     47     48     49     50     51     52     53     54     55     56     57     58     59     60     61     62     63     64     65     66     67     68     69     70     71     72     73     74     75     76     77     78     79     80     81     82     83     84     85     86     87     88     89     90     91     92     93     94     95     96     97     98     
     +-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+

In [ ]:
print('''
       0     1     2     3
    +-----+-----+-----+-----+
 0  | [ ] |  1  |  1  |  1  |
    +-----+-----+-----+-----+
 1  |  1  |  2  |  x  |  2  |
    +-----+-----+-----+-----+
 2  |  x  |  3  |  3  |  x  |
    +-----+-----+-----+-----+
 3  |  2  |  x  |  3  |  1  |
    +-----+-----+-----+-----+
''')